In [5]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import TimeSeriesSplit

DATA_FILE = "../../data/model_data_success.csv"
CUTOFF_DATE = "2017-11-18"
N_VALIDATION_SPLITS = 3
TARGET = "success"

movies = pd.read_csv(DATA_FILE, parse_dates=["release_date"])
movies = movies.sort_values("release_date", kind="stable").reset_index(drop=True)
y = movies[TARGET].copy()

train = movies["release_date"].lt(pd.Timestamp(CUTOFF_DATE))
test = ~train
y_train, y_test = y.loc[train], y.loc[test]

training_dates = movies.loc[train, "release_date"]
unique_dates = np.sort(training_dates.unique())
feature_splits = []
for earlier, later in TimeSeriesSplit(n_splits=N_VALIDATION_SPLITS).split(unique_dates):
    fit_rows = training_dates.index[training_dates.isin(unique_dates[earlier])]
    validation_rows = training_dates.index[training_dates.isin(unique_dates[later])]
    feature_splits.append((fit_rows, validation_rows))

print(f"{len(movies):,} movies | train {train.sum():,} | test {test.sum():,} | folds {len(feature_splits)}")
print("Hit rate:", round(y.mean(), 3))
print(list(movies.columns))

6,919 movies | train 5,535 | test 1,384 | folds 3
Hit rate: 0.362
['tmdb_id', 'imdb_id', 'title_tmdb', 'release_date', 'runtime', 'log_budget', 'release_year', 'n_genres', 'imdb_director_count', 'dir_prior_success_rate_smoothed', 'dir_prior_rating_smoothed', 'dir_prior_films_log1p', 'dir_years_active_log1p', 'dir_prior_known_films_log1p', 'dir_prior_rated_films_log1p', 'lead_actor_films_5yr_backfilled_log1p', 'company_films_5yr_log1p', 'franchise_prior_films_log1p', 'us_certification', 'release_month', 'is_sequel', 'is_english', 'dir_first_film', 'dir_no_track_record', 'backfill_unavailable', 'dir_no_rating_history', 'genre_drama', 'genre_comedy', 'genre_thriller', 'genre_action', 'genre_adventure', 'genre_crime', 'genre_romance', 'genre_horror', 'genre_science_fiction', 'genre_family', 'genre_fantasy', 'genre_mystery', 'genre_animation', 'genre_history', 'genre_war', 'genre_music', 'genre_western', 'genre_documentary', 'success', 'vote_average']


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

CATEGORICAL_FEATURES = ["us_certification", "release_month"]
BINARY_FEATURES = ["is_sequel", "is_english", "dir_first_film", "dir_no_track_record",
                   "backfill_unavailable", "dir_no_rating_history"] + [
    c for c in movies if c.startswith("genre_")]

basics = ["runtime", "log_budget", "us_certification", "release_month", "is_sequel", "is_english"]
feature_versions = {
    "Basics": basics,
    "Basics + four genres": basics + ["genre_drama", "genre_comedy", "genre_action", "genre_horror"],
    "Basics + director success": basics + ["dir_prior_success_rate_smoothed",
                                           "dir_prior_films_log1p", "dir_no_track_record"],
    "Basics + actor/company/franchise": basics + [
        "lead_actor_films_5yr_backfilled_log1p", "backfill_unavailable",
        "company_films_5yr_log1p", "franchise_prior_films_log1p"],
}

comparison_features = list(dict.fromkeys(c for cols in feature_versions.values() for c in cols))
comparison_X = movies[comparison_features].copy()
for column in CATEGORICAL_FEATURES:
    comparison_X[column] = comparison_X[column].map(lambda v: str(v) if pd.notna(v) else np.nan)

def make_feature_model(columns):
    numeric = [c for c in columns if c not in CATEGORICAL_FEATURES + BINARY_FEATURES]
    categorical = [c for c in CATEGORICAL_FEATURES if c in columns]
    binary = [c for c in columns if c in BINARY_FEATURES]
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                                  ("scaler", StandardScaler())]), numeric),
            ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                      ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))]), categorical),
            ("binary", "passthrough", binary),
        ])),
    ])

print({name: len(cols) for name, cols in feature_versions.items()})

{'Basics': 6, 'Basics + four genres': 10, 'Basics + director success': 9, 'Basics + actor/company/franchise': 10}


In [7]:
from itertools import product
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

KNN_GRID = {
    "n_neighbors": [10, 20, 30, 40, 50, 75, 100, 150],
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}
THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
MIN_VALIDATION_PRECISION = 0.70

def make_knn_model(columns, **params):
    pipeline = make_feature_model(columns)
    pipeline.steps.append(("model", KNeighborsClassifier(**params)))
    return pipeline

knn_rows = []
for version, columns in feature_versions.items():
    for k, w, p in product(KNN_GRID["n_neighbors"], KNN_GRID["weights"], KNN_GRID["p"]):
        probs = []
        for fit_rows, validation_rows in feature_splits:
            knn = make_knn_model(columns, n_neighbors=k, weights=w, p=p)
            knn.fit(comparison_X.loc[fit_rows], y.loc[fit_rows])
            probs.append(knn.predict_proba(comparison_X.loc[validation_rows])[:, 1])
        for t in THRESHOLDS:
            fold_p, fold_r, fold_f = [], [], []
            for (fit_rows, validation_rows), prob in zip(feature_splits, probs):
                pred = (prob >= t).astype(int)
                actual = y.loc[validation_rows]
                fold_p.append(precision_score(actual, pred, zero_division=0))
                fold_r.append(recall_score(actual, pred, zero_division=0))
                fold_f.append(f1_score(actual, pred, zero_division=0))
            knn_rows.append({"Version": version, "n_neighbors": k, "weights": w, "p": p,
                             "threshold": t, "Val_precision": np.mean(fold_p),
                             "Val_recall": np.mean(fold_r), "Val_F1": np.mean(fold_f)})

knn_search = pd.DataFrame(knn_rows)
qualifying = knn_search[knn_search["Val_precision"] >= MIN_VALIDATION_PRECISION]
print("Candidates:", len(knn_search), "| meeting 70% precision:", len(qualifying))

if len(qualifying):
    best_knn = qualifying.sort_values("Val_F1", ascending=False).iloc[0]
    display(qualifying.sort_values("Val_F1", ascending=False).head(10).round(4))
else:
    best_knn = knn_search.sort_values("Val_precision", ascending=False).iloc[0]
    print("No candidate met the precision goal. Highest-precision candidate shown for reference only.")
    display(knn_search.sort_values("Val_precision", ascending=False).head(10).round(4))
print("Best KNN:", best_knn.to_dict())

Candidates: 1024 | meeting 70% precision: 53


,Version,n_neighbors,weights,p,threshold,Val_precision,Val_recall,Val_F1
1020,Basics + actor/company/franchise,150,distance,2,0.50,0.7975,0.1303,0.2024
1012,Basics + actor/company/franchise,150,distance,1,0.50,0.8358,0.0881,0.1471
483,Basics + four genres,150,uniform,1,0.45,0.7600,0.0806,0.1431
420,Basics + four genres,75,uniform,1,0.50,0.7226,0.0788,0.1409
428,Basics + four genres,75,uniform,2,0.50,0.7038,0.0791,0.1406
460,Basics + four genres,100,uniform,2,0.50,0.7520,0.0696,0.1254
397,Basics + four genres,50,uniform,2,0.55,0.7334,0.0681,0.1232
437,Basics + four genres,75,distance,1,0.55,0.7108,0.0670,0.1219
989,Basics + actor/company/franchise,100,distance,2,0.55,0.8080,0.0728,0.1218
452,Basics + four genres,100,uniform,1,0.50,0.7759,0.0608,0.1115


Best KNN: {'Version': 'Basics + actor/company/franchise', 'n_neighbors': 150, 'weights': 'distance', 'p': 2, 'threshold': 0.5, 'Val_precision': 0.7974926934317796, 'Val_recall': 0.1303169176982842, 'Val_F1': 0.2024246777471533}


In [8]:
final_knn = make_knn_model(
    feature_versions[best_knn["Version"]],
    n_neighbors=int(best_knn["n_neighbors"]),
    weights=best_knn["weights"],
    p=int(best_knn["p"]),
)
final_knn.fit(comparison_X.loc[train], y_train)
test_prob = final_knn.predict_proba(comparison_X.loc[test])[:, 1]
test_pred = (test_prob >= best_knn["threshold"]).astype(int)

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

knn_test_results = pd.DataFrame([{
    "Model": "KNN (" + best_knn["Version"] + ", k=" + str(int(best_knn["n_neighbors"])) + ")",
    "Threshold": best_knn["threshold"],
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "Hit F1": f1_score(y_test, test_pred, zero_division=0),
    "Accuracy": accuracy_score(y_test, test_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, test_pred),
}])
display(knn_test_results.round(4))
print(confusion_matrix(y_test, test_pred))

,Model,Threshold,Precision,Recall,Hit F1,Accuracy,Balanced accuracy
0,"KNN (Basics + actor/company/franchise, k=150)",0.5,0.5565,0.28,0.3725,0.6763,0.5817


[[803 106]
 [342 133]]


# **PCA**

In [9]:
from sklearn.decomposition import PCA

PCA_VARIANCE = 0.70

def make_pca_knn_model(columns, **params):
    base = make_feature_model(columns)
    return Pipeline([
        ("preprocessor", base.named_steps["preprocessor"]),
        ("pca", PCA(n_components=PCA_VARIANCE)),
        ("model", KNeighborsClassifier(**params)),
    ])

pca_knn = make_pca_knn_model(
    feature_versions[best_knn["Version"]],
    n_neighbors=int(best_knn["n_neighbors"]),
    weights=best_knn["weights"],
    p=int(best_knn["p"]),
)
pca_knn.fit(comparison_X.loc[train], y_train)

print(pca_knn.named_steps["pca"].n_components_)
print(pca_knn.named_steps["pca"].explained_variance_ratio_.cumsum().round(3))

5
[0.277 0.439 0.585 0.7   0.771]


In [10]:
pca_prob = pca_knn.predict_proba(comparison_X.loc[test])[:, 1]
pca_pred = (pca_prob >= best_knn["threshold"]).astype(int)

pca_row = pd.DataFrame([{
    "Model": "KNN + PCA (70%)",
    "Threshold": best_knn["threshold"],
    "Precision": precision_score(y_test, pca_pred, zero_division=0),
    "Recall": recall_score(y_test, pca_pred, zero_division=0),
    "Hit F1": f1_score(y_test, pca_pred, zero_division=0),
    "Accuracy": accuracy_score(y_test, pca_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, pca_pred),
}])
display(pd.concat([knn_test_results, pca_row], ignore_index=True).round(4))
print(confusion_matrix(y_test, pca_pred))

,Model,Threshold,Precision,Recall,Hit F1,Accuracy,Balanced accuracy
0,"KNN (Basics + actor/company/franchise, k=150)",0.5,0.5565,0.2800,0.3725,0.6763,0.5817
1,KNN + PCA (70%),0.5,0.5745,0.2842,0.3803,0.6821,0.5871


[[809 100]
 [340 135]]


In [13]:
params = dict(n_neighbors=int(best_knn["n_neighbors"]),
              weights=best_knn["weights"], p=int(best_knn["p"]))
columns = feature_versions[best_knn["Version"]]

pca_rows = []
for variance in [0.70, 0.80, 0.90, 0.95, None]:
    label = "No PCA" if variance is None else f"PCA {int(variance * 100)}%"
    probs = []
    for fit_rows, validation_rows in feature_splits:
        if variance is None:
            model = make_knn_model(columns, **params)
        else:
            base = make_feature_model(columns)
            model = Pipeline([
                ("preprocessor", base.named_steps["preprocessor"]),
                ("pca", PCA(n_components=variance)),
                ("model", KNeighborsClassifier(**params)),
            ])
        model.fit(comparison_X.loc[fit_rows], y.loc[fit_rows])
        probs.append(model.predict_proba(comparison_X.loc[validation_rows])[:, 1])
    for t in THRESHOLDS:
        fold_p, fold_r, fold_f = [], [], []
        for (fit_rows, validation_rows), prob in zip(feature_splits, probs):
            pred = (prob >= t).astype(int)
            actual = y.loc[validation_rows]
            fold_p.append(precision_score(actual, pred, zero_division=0))
            fold_r.append(recall_score(actual, pred, zero_division=0))
            fold_f.append(f1_score(actual, pred, zero_division=0))
        pca_rows.append({"Setting": label, "threshold": t, "Val_precision": np.mean(fold_p),
                         "Val_recall": np.mean(fold_r), "Val_F1": np.mean(fold_f)})

pca_search = pd.DataFrame(pca_rows)
qualifying = pca_search[pca_search["Val_precision"] >= MIN_VALIDATION_PRECISION]
print(qualifying.groupby("Setting").size().rename("candidates meeting 70% precision"))
display(qualifying.sort_values("Val_F1", ascending=False).groupby("Setting").head(1).round(4))

Setting
No PCA     1
PCA 70%    1
PCA 80%    1
PCA 90%    1
Name: candidates meeting 70% precision, dtype: int64


,Setting,threshold,Val_precision,Val_recall,Val_F1
20,PCA 90%,0.50,0.7121,0.1336,0.2066
36,No PCA,0.50,0.7975,0.1303,0.2024
5,PCA 70%,0.55,0.7508,0.0620,0.1047
13,PCA 80%,0.55,0.8873,0.0586,0.0991


**Adding PCA**

We also tested PCA before KNN, keeping enough components to explain 70%, 80%, 90%, or 95% of the variance, with k, weighting, distance, and features held at the selected values. Under the project's rule (at least 70% mean validation hit precision, then highest hit F1), PCA 90% scored a validation hit F1 of 0.207 versus 0.202 without PCA, a negligible gap that comes with a much thinner precision margin (71.2% vs 79.7%). PCA 70% and 80% scored about 0.10, and PCA 95% had no qualifying candidate. On the test set, PCA at 70% reached a hit F1 of 0.380 versus 0.3725 without it (precision 57.5% vs 55.7%), a difference of a handful of movies. PCA did not improve predictions, so the final KNN uses the original features without PCA.